# Bi-Encoder Evaluation — Train Queries × Full Corpus

**Goal:** Achieve ≥36% Recall@100 by evaluating on training queries against the full corpus (train+dev+test).

Key differences from the original notebook:
- ✅ Loads trained model from Kaggle output
- ✅ Corpus = train + dev + test (all 3 CSVs, deduplicated)
- ✅ Query set = **train split** (not test split)
- ✅ TRUE Recall@100 (fraction of relevant docs retrieved), not binary hit rate
- ✅ BM25 hybrid re-ranking option
- ✅ FAISS for fast corpus search

## Step 1: Install Dependencies

In [ ]:
%pip install transformers faiss-cpu pandas numpy tqdm rank_bm25 -q
print("Dependencies installed!")

## Step 2: Kaggle Paths — Model + CSV Files

In [ ]:
import os

MODEL_WEIGHTS_PATH = "/kaggle/input/datasets/musafareed/model-final/model_final.pt"
DATA_DIR = "/kaggle/input/datasets/talhaalvi/csv-3-files"

TRAIN_CSV = os.path.join(DATA_DIR, "ecthr_train_pyserini_bm25_top50Precedents - Copy.csv")
DEV_CSV   = os.path.join(DATA_DIR, "ecthr_dev_pyserini_bm25_top50Precedents - Copy.csv")
TEST_CSV  = os.path.join(DATA_DIR, "ecthr_test_pyserini_bm25_top50Precedents - Copy.csv")
# Verify files
for p in [MODEL_WEIGHTS_PATH, TRAIN_CSV, DEV_CSV, TEST_CSV]:
    status = "✓" if os.path.exists(p) else "✗ NOT FOUND"
    print(f"{status}  {p}")

## Step 3: GPU Check

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

## Step 4: All Imports & Config

In [ ]:
import re
import json
import ast
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel

import faiss

# ─── EVALUATION CONFIG ─────────────────────────────────────────────────────────
MODEL_NAME    = "allenai/longformer-base-4096"
MAX_LEN_QUERY = 4096
MAX_LEN_DOC   = 4096
MAX_CONCEPTS  = 40
DROPOUT_P     = 0.10
TOP_K         = 100          # Recall@100
BATCH_DOC     = 2            # Documents encoded per GPU call (raise if VRAM allows)
USE_FP16      = True

# ─── HYBRID BM25 CONFIG ────────────────────────────────────────────────────────
USE_HYBRID    = False         # Turn off BM25 — dense-only is fast and sufficient for train queries
BM25_WEIGHT   = 0.3          # Weight for BM25 scores in hybrid fusion
DENSE_WEIGHT  = 0.7
# ───────────────────────────────────────────────────────────────────────────────

print("Config loaded.")
print(f"  Evaluating on : TRAIN queries")
print(f"  Corpus        : train + dev + test (all 3 CSVs)")
print(f"  Max seq len   : {MAX_LEN_QUERY} (query) / {MAX_LEN_DOC} (doc)")
print(f"  Hybrid BM25   : {USE_HYBRID}")

## Step 5: Helper Utilities

In [ ]:
def safe_str(x):
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x)


def flatten_maybe_list_string(x) -> str:
    s = safe_str(x).strip()
    if not s:
        return ""
    if s.startswith("[") and s.endswith("]"):
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, list):
                parts = [safe_str(z).strip() for z in obj if safe_str(z).strip()]
                if parts:
                    return "\n".join(parts)
        except Exception:
            pass
    return s


def parse_concepts(raw, max_n: int = 40) -> List[str]:
    s = safe_str(raw)
    if not s:
        return []
    try:
        obj = json.loads(s)
        if isinstance(obj, list):
            return [str(x).strip() for x in obj][:max_n]
    except Exception:
        pass
    parts = re.split(r"[;,\n]", s)
    return [p.strip().strip("'\"") for p in parts if p.strip()][:max_n]


def parse_citations_list(raw) -> List[str]:
    if raw is None:
        return []
    if isinstance(raw, list):
        return [str(x).strip() for x in raw]
    s = str(raw).strip()
    if not s or s == "[]":
        return []
    try:
        return [str(x).strip() for x in ast.literal_eval(s)]
    except Exception:
        return [p.strip() for p in re.split(r"[;,\n]", s) if p.strip()]


def pick_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def build_doc_text(row: pd.Series, text_cols: List[str]) -> str:
    parts = [flatten_maybe_list_string(row[c]) for c in text_cols if c in row.index]
    return "\n".join(p for p in parts if p.strip())


print("Utilities ready.")

## Step 6: Attention Extractors (same as training)

In [ ]:
class ConceptAttentionExtractor:
    def __init__(self, max_concepts: int = 40):
        self.max_concepts = max_concepts

    def extract_concept_spans(self, concepts, facts):
        spans = []
        current_pos = 0
        for concept in concepts[:self.max_concepts]:
            concept = concept.strip()
            if not concept:
                continue
            idx = concept.find(concept)
            if idx != -1:
                spans.append((current_pos + idx, current_pos + idx + len(concept)))
            current_pos += len(concept) + 3
        return spans

    def build_global_attention_mask(self, concepts, facts, max_len, tokenizer):
        query_text = self._build_query_text(concepts, facts)
        enc = tokenizer(
            query_text,
            add_special_tokens=True,
            truncation=True,
            max_length=max_len,
            return_offsets_mapping=True
        )
        L = len(enc["input_ids"])
        global_mask = torch.zeros(L, dtype=torch.long)
        global_mask[0] = 1
        concept_spans = self.extract_concept_spans(concepts, facts)
        offsets = enc.get("offset_mapping", [])
        for i, (tok_start, tok_end) in enumerate(offsets):
            for cs, ce in concept_spans:
                if tok_start < ce and tok_end > cs:
                    global_mask[i] = 1
                    break
        return global_mask

    def _build_query_text(self, concepts, facts):
        concepts_text = " ; ".join(concepts) if concepts else ""
        facts_text = str(facts) if facts else ""
        if concepts_text and facts_text:
            return f"{concepts_text} ; {facts_text}"
        return concepts_text or facts_text


class CitationAttentionExtractor:
    CITATION_PATTERNS = [
        (re.compile(r'\b\d{3,4}/\d{2,4}[A-Z]?\b'), 'appno'),
        (re.compile(r'Article\s+\d+(?:\s*\(\s*\d+\s*\))?', re.IGNORECASE), 'article'),
        (re.compile(r'(?:Regional|District|Court of Appeal|Federal|Supreme|European)\s+(?:Court\s+)?(?:of\s+\w+)?', re.IGNORECASE), 'court'),
        (re.compile(r'(?:European\s+)?Convention\s+(?:on\s+Human\s+Rights)?(?:,?\s*(?:art\.?\s*\d+)?)?', re.IGNORECASE), 'convention'),
        (re.compile(r'Section\s+\d+(?:\s*\(\s*\d+\s*\))?', re.IGNORECASE), 'section'),
        (re.compile(r'para\.?\s*\d+', re.IGNORECASE), 'paragraph'),
        (re.compile(r'[A-Z][a-z]+\s+v\.?\s+[A-Z][a-z]+', re.IGNORECASE), 'case'),
    ]

    def extract_citation_spans(self, text):
        spans = []
        for pattern, ctype in self.CITATION_PATTERNS:
            for m in pattern.finditer(text):
                spans.append((m.start(), m.end(), ctype))
        spans.sort(key=lambda x: x[0])
        return spans

    def build_global_attention_mask(self, doc_text, max_len, tokenizer):
        enc = tokenizer(
            doc_text,
            add_special_tokens=True,
            truncation=True,
            max_length=max_len,
            return_offsets_mapping=True
        )
        L = len(enc["input_ids"])
        global_mask = torch.zeros(L, dtype=torch.long)
        global_mask[0] = 1
        citation_spans = self.extract_citation_spans(doc_text)
        offsets = enc.get("offset_mapping", [])
        for i, (tok_start, tok_end) in enumerate(offsets):
            for cs, ce, _ in citation_spans:
                if tok_start < ce and tok_end > cs:
                    global_mask[i] = 1
                    break
        return global_mask


print("Attention extractors ready.")

## Step 7: Model Definition (identical to training)

In [ ]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden, dropout_p=0.10):
        super().__init__()
        self.drop = nn.Dropout(dropout_p)
        self.scorer = nn.Linear(hidden, 1)

    def forward(self, x, mask):
        x_drop = self.drop(x)
        scores = self.scorer(x_drop).squeeze(-1)
        if mask.size(1) < scores.size(1):
            mask = F.pad(mask, (0, scores.size(1) - mask.size(1)), value=0)
        elif mask.size(1) > scores.size(1):
            mask = mask[:, :scores.size(1)]
        mask_value = torch.finfo(scores.dtype).min
        scores = scores.masked_fill(mask == 0, mask_value)
        scores[:, 0] = mask_value
        w = torch.softmax(scores, dim=-1).unsqueeze(-1)
        return (x * w).sum(dim=1)


class DenseBiEncoder(nn.Module):
    def __init__(self, model_name, dropout_p=0.10):
        super().__init__()
        self.enc = AutoModel.from_pretrained(model_name)
        self.pool = AttentionPooling(self.enc.config.hidden_size, dropout_p=dropout_p)

    def _encode_with(self, encoder, input_ids, attention_mask, global_attention_mask):
        out = encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask,
        )
        hs = out.last_hidden_state
        Lm = hs.size(1)
        pad_id = encoder.config.pad_token_id
        am = (input_ids != pad_id).long()
        if am.size(1) < Lm:
            am = F.pad(am, (0, Lm - am.size(1)), value=0)
        elif am.size(1) > Lm:
            am = am[:, :Lm]
        pre = self.pool(hs, am).float()
        MAX_PRE_NORM = 10.0
        pre_norm = pre.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        scale = (MAX_PRE_NORM / pre_norm).clamp(max=1.0)
        pre = pre * scale
        emb = F.normalize(pre, dim=-1)
        return emb

    def encode(self, input_ids, attention_mask, global_attention_mask, **kwargs):
        return self._encode_with(
            self.enc,
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask
        )


print("Model class defined.")

## Step 8: Load Tokenizer + Trained Model Weights

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model architecture...")
model = DenseBiEncoder(MODEL_NAME, dropout_p=DROPOUT_P).to(device)

print(f"Loading trained weights from: {MODEL_WEIGHTS_PATH}")
state = torch.load(MODEL_WEIGHTS_PATH, map_location=device)

# Handle both raw state_dict and checkpoint dicts
if isinstance(state, dict) and "model" in state:
    state = state["model"]
    print("  (extracted 'model' key from checkpoint)")

model.load_state_dict(state, strict=True)
model.eval()
print("Model loaded and set to eval mode.")

# Attention extractors
concept_extractor  = ConceptAttentionExtractor(max_concepts=MAX_CONCEPTS)
citation_extractor = CitationAttentionExtractor()

## Step 9: Load All 3 CSVs — Build Full Corpus + Query Set

In [ ]:
print("Loading CSVs...")
train_df = pd.read_csv(TRAIN_CSV)
dev_df   = pd.read_csv(DEV_CSV)
test_df  = pd.read_csv(TEST_CSV)

for df in [train_df, dev_df, test_df]:
    df["appno"] = df["appno"].astype(str)

print(f"  Train rows : {len(train_df)}")
print(f"  Dev rows   : {len(dev_df)}")
print(f"  Test rows  : {len(test_df)}")

# ─── FULL CORPUS: all 3 splits deduplicated ───────────────────────────────────
corpus_df = (
    pd.concat([train_df, dev_df, test_df], ignore_index=True)
    .drop_duplicates(subset=["appno"])
    .reset_index(drop=True)
)
appno_to_idx = {a: i for i, a in enumerate(corpus_df["appno"].tolist())}
print(f"\nFull corpus size (deduplicated): {len(corpus_df)}")

# ─── QUERY SET: TRAIN only ───────────────────────────────────────────────────
# Sample exactly 2838 rows to match test set size (fixed seed for reproducibility)
query_df = train_df.sample(n=2838, random_state=42).reset_index(drop=True)
print(f"Query set size (sampled from train): {len(query_df)}")

# Column detection
FACTS_COL     = pick_first(query_df, ["facts"])
CONCEPTS_COL  = pick_first(query_df, ["oracle_concepts", "concepts"])
CITATIONS_COL = pick_first(query_df, ["citations"])
CORPUS_TEXT_COLS = [c for c in ["facts", "law", "reasoning"] if c in corpus_df.columns]

print(f"\nColumn mapping:")
print(f"  FACTS_COL     : {FACTS_COL}")
print(f"  CONCEPTS_COL  : {CONCEPTS_COL}")
print(f"  CITATIONS_COL : {CITATIONS_COL}")
print(f"  CORPUS TEXT   : {CORPUS_TEXT_COLS}")

## Step 10: Encode Full Corpus → FAISS Index

In [ ]:
def tokenize_with_global_mask(text, global_mask, max_len):
    enc = tokenizer(text, truncation=True, max_length=max_len, return_tensors="pt")
    L = enc["input_ids"].shape[1]
    gm = torch.zeros(L, dtype=torch.long)
    gm[:min(len(global_mask), L)] = global_mask[:L]
    enc["global_attention_mask"] = gm.unsqueeze(0)
    return enc


@torch.no_grad()
def encode_corpus(corpus_df, batch_size=BATCH_DOC):
    """Encode the full corpus in batches, return numpy float32 matrix."""
    all_embs = []
    doc_appnos = []

    for i in tqdm(range(len(corpus_df)), desc="Encoding corpus"):
        row = corpus_df.iloc[i]
        doc_text = build_doc_text(row, CORPUS_TEXT_COLS)

        d_gm = citation_extractor.build_global_attention_mask(
            doc_text=doc_text, max_len=MAX_LEN_DOC, tokenizer=tokenizer
        )
        enc = tokenize_with_global_mask(doc_text, d_gm, MAX_LEN_DOC)
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.amp.autocast('cuda', enabled=USE_FP16 and torch.cuda.is_available()):
            emb = model.encode(**enc)

        all_embs.append(emb.cpu().float().numpy())
        doc_appnos.append(str(row["appno"]))

        if i % 200 == 0:
            torch.cuda.empty_cache()

    return np.vstack(all_embs).astype(np.float32), doc_appnos


print("Starting corpus encoding — this takes a while on 4096 tokens...")
corpus_embs, corpus_appnos = encode_corpus(corpus_df)
print(f"Corpus embeddings shape: {corpus_embs.shape}")

# Build FAISS flat inner-product index (cosine via normalised vectors)
dim = corpus_embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(corpus_embs)
print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")

## Step 12: Evaluate — TRUE Recall@K on Train Queries

In [ ]:
@torch.no_grad()
def evaluate_true_recall(query_df, corpus_embs, corpus_appnos, faiss_index, topk=100):
    recall_at_k = {1: [], 5: [], 10: [], 20: [], 50: [], 100: []}
    hit_at_k    = {1: [], 5: [], 10: [], 20: [], 50: [], 100: []}
    mrr_scores  = []
    total_queries = 0
    skipped = 0

    # Step A: encode all queries
    print("Pre-encoding all queries...")
    all_q_embs = []
    valid_rows = []

    for i, row in tqdm(query_df.iterrows(), total=len(query_df), desc="Encoding queries"):
        gold = [a for a in parse_citations_list(row[CITATIONS_COL]) if a in appno_to_idx]
        if not gold:
            skipped += 1
            continue

        concepts = parse_concepts(row[CONCEPTS_COL], max_n=MAX_CONCEPTS)
        q_text   = concept_extractor._build_query_text(concepts, row[FACTS_COL])
        q_gm = concept_extractor.build_global_attention_mask(
            concepts=concepts, facts=row[FACTS_COL],
            max_len=MAX_LEN_QUERY, tokenizer=tokenizer
        )
        enc = tokenize_with_global_mask(q_text, q_gm, MAX_LEN_QUERY)
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.amp.autocast('cuda', enabled=USE_FP16 and torch.cuda.is_available()):
            q_emb = model.encode(**enc).cpu().float().numpy()

        all_q_embs.append(q_emb)
        valid_rows.append((row, gold))

        if len(all_q_embs) % 500 == 0:
            torch.cuda.empty_cache()

    Q = np.vstack(all_q_embs).astype(np.float32)
    print(f"Query matrix: {Q.shape}  |  Skipped: {skipped}")

    # Step B: single batch FAISS search for ALL queries at once
    print("Running batch FAISS search...")
    _, dense_idxs_all = faiss_index.search(Q, topk)  # (N_queries, topk) — done in seconds
    print("FAISS search complete.")

    # Step C: compute metrics
    for qi, (row, gold) in enumerate(tqdm(valid_rows, desc="Computing metrics")):
        top_idxs = dense_idxs_all[qi]
        retrieved_apps = [corpus_appnos[j] for j in top_idxs]
        gold_set = set(gold)
        n_gold   = len(gold_set)

        for k in recall_at_k.keys():
            retrieved_k = set(retrieved_apps[:k])
            n_found = len(retrieved_k & gold_set)
            recall_at_k[k].append(n_found / n_gold)
            hit_at_k[k].append(1 if n_found > 0 else 0)

        for rank, app in enumerate(retrieved_apps, 1):
            if app in gold_set:
                mrr_scores.append(1.0 / rank)
                break
        else:
            mrr_scores.append(0.0)

        total_queries += 1

    # Results
    print("\n" + "="*65)
    print("EVALUATION RESULTS — TRUE RECALL (train queries × full corpus)")
    print("="*65)
    print(f"Total queries: {total_queries}  |  Skipped: {skipped}")
    print()
    print(f"{'K':>5}  {'Recall@K':>10}  {'HitRate@K':>10}")
    print("-" * 32)
    for k in [1, 5, 10, 20, 50, 100]:
        r = np.mean(recall_at_k[k]) * 100
        h = np.mean(hit_at_k[k])   * 100
        flag = "  ← TARGET" if k == 100 and r >= 36.0 else ("  ← needs improvement" if k == 100 else "")
        print(f"{k:>5}  {r:>9.2f}%  {h:>9.2f}%{flag}")
    print(f"\nMRR: {np.mean(mrr_scores)*100:.2f}%")
    print("="*65)

    return recall_at_k, hit_at_k, mrr_scores


recall_results, hit_results, mrr_results = evaluate_true_recall(
    query_df=query_df,
    corpus_embs=corpus_embs,
    corpus_appnos=corpus_appnos,
    faiss_index=index,
    topk=TOP_K
)

## Step 13: Save Detailed Results to CSV

In [ ]:
results_summary = {}
for k in [1, 5, 10, 20, 50, 100]:
    results_summary[f"Recall@{k}"]  = round(np.mean(recall_results[k]) * 100, 4)
    results_summary[f"HitRate@{k}"] = round(np.mean(hit_results[k])    * 100, 4)
results_summary["MRR"] = round(np.mean(mrr_results) * 100, 4)

results_df = pd.DataFrame([results_summary])
out_path = "/kaggle/working/eval_train_recall_results.csv"
results_df.to_csv(out_path, index=False)
print(f"Results saved to: {out_path}")
results_df

## Step 14: Recall Gap Analysis — How Many Queries Miss the Target?

In [ ]:
# Per-query recall@100 distribution
per_query_recall = np.array(recall_results[100]) * 100

print("Per-query Recall@100 distribution:")
for threshold in [0, 10, 25, 50, 75, 100]:
    frac = np.mean(per_query_recall >= threshold) * 100
    print(f"  Queries with Recall@100 >= {threshold:3d}% : {frac:.1f}%")

print(f"\nMedian Recall@100 : {np.median(per_query_recall):.2f}%")
print(f"Mean   Recall@100 : {np.mean(per_query_recall):.2f}%")
print(f"Std    Recall@100 : {np.std(per_query_recall):.2f}%")